# 08_WebcamWide — Test webcam bản TCN `wide45_lownoise`

Chỉ chạy model, **không train**. Dùng checkpoint `Data_321/checkpoints_v2/wide45_lownoise/TCN.pt`.

**Cách chạy:** kích hoạt môi trường `C:\envs\train-gpu\Scripts\activate.bat`, mở notebook bằng kernel `train-gpu`, chạy lần lượt cell 1 → 7. Cell 7 mở cửa sổ webcam.

**Phím trong cửa sổ webcam:** `q` thoát | `[` giảm ngưỡng tin cậy | `]` tăng ngưỡng (mỗi lần 0.05).

**Giao diện:** khung tay + khung xương đổi màu theo lớp, **paper = đỏ, rock = xanh dương, scissors = xanh lá**. Chữ luôn là **chữ đen trên nền màu**. Khi độ tin cậy chưa đạt ngưỡng, khung xám và có dấu `?`. Góc trên trái hiện FPS và ngưỡng, góc dưới phải là chú thích màu. Chữ không dấu vì OpenCV không vẽ được chữ có dấu.

**Cách đưa dữ liệu vào model:** webcam khoảng 30 fps nhưng model train ở 120 fps, nên mỗi khung webcam mới được nội suy tuyến tính thành các khung 120 fps (chỉ dùng khung trước và khung hiện tại, không cần nhìn trước). Đặc trưng 126 số dựng giống hệt 04_Trainv2.

In [1]:
#cell 1: đường dẫn, kiểm tra các file cần có, bật CUDA
import time, collections
import numpy as np, pandas as pd, torch, cv2
import mediapipe as mp
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không chạy trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
TASK = BASE / "hand_landmarker.task"
CKPT = BASE / "checkpoints_v2" / "wide45_lownoise" / "TCN.pt"
for p in (BASE / "manifest.csv", TASK, CKPT):
    assert p.exists(), f"Không thấy: {p}"
print("OK | GPU:", torch.cuda.get_device_name(0))

OK | GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [2]:
#cell 2: kiến trúc TCN (giống 04_Trainv2), nạp checkpoint wide45_lownoise, làm nóng GPU
import torch.nn as nn, torch.nn.functional as F

class Block(nn.Module):
    def __init__(s, c, d):
        super().__init__(); s.pad = 2 * d; s.conv = nn.Conv1d(c, c, 3, dilation=d); s.drop = nn.Dropout(0.1)
    def forward(s, x):
        return x + s.drop(F.relu(s.conv(F.pad(x, (s.pad, 0)))))

class TCN(nn.Module):
    def __init__(s, c=64):
        super().__init__(); s.inp = nn.Linear(126, c)
        s.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        s.out = nn.Linear(c, 3)
    def forward(s, x):
        return s.out(s.blocks(s.inp(x).transpose(1, 2)).transpose(1, 2))

CLASSES = ["paper", "rock", "scissors"]          # nhãn 0, 1, 2 theo thứ tự chữ cái
net = TCN().to(dev)
ck = torch.load(CKPT, map_location=dev, weights_only=False)
net.load_state_dict(ck["w"]); net.eval()
with torch.no_grad(): net(torch.zeros(1, 128, 126, device=dev))   # khởi động CUDA để khung đầu không bị giật
print(f"Đã nạp {CKPT.name} | epoch {ck['epoch']} | cấu hình: {ck.get('cfg')}")

Đã nạp TCN.pt | epoch 49 | cấu hình: {'rot': (40, 45, 40), 'noise': 0.005}


In [3]:
#cell 3: tự kiểm tra loại tọa độ đã dùng lúc train (ảnh chuẩn hóa hay world) để webcam dùng đúng loại
m = pd.read_csv(BASE / "manifest.csv")
lms = np.concatenate([np.load(BASE / p)["lm"].reshape(-1, 3) for p in m.path.iloc[:5]])
USE_WORLD = bool(np.nanmin(lms[:, :2]) < 0 and np.nanmax(np.abs(lms)) < 0.3)
print(f"Tọa độ x trong file train: [{np.nanmin(lms[:, 0]):.3f}, {np.nanmax(lms[:, 0]):.3f}]"
      f" -> webcam dùng tọa độ {'world' if USE_WORLD else 'ảnh chuẩn hóa (0-1)'}")

Tọa độ x trong file train: [0.278, 0.828] -> webcam dùng tọa độ ảnh chuẩn hóa (0-1)


In [4]:
#cell 4: MediaPipe HandLandmarker (CPU, chế độ VIDEO, num_hands=1) và hàm lấy tọa độ tay
landmarker = mp.tasks.vision.HandLandmarker.create_from_options(mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO, num_hands=1))
T0 = time.perf_counter()     # mốc thời gian dùng chung: timestamp của MediaPipe phải luôn tăng, kể cả khi chạy lại cell 7

def detect(frame_bgr, ts_ms):
    """Trả về (img_lm, model_lm), mỗi cái (21,3): img_lm để vẽ, model_lm đưa vào model. Mất tay -> (None, None)."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res = landmarker.detect_for_video(mp.Image(mp.ImageFormat.SRGB, rgb), ts_ms)
    if not res.hand_landmarks:
        return None, None
    arr = lambda pts: np.array([[p.x, p.y, p.z] for p in pts], np.float32)
    img_lm = arr(res.hand_landmarks[0])
    return img_lm, (arr(res.hand_world_landmarks[0]) if USE_WORLD else img_lm)
print("Đã tạo HandLandmarker (CPU).")

Đã tạo HandLandmarker (CPU).


In [5]:
#cell 5: nội suy webcam -> 120 fps, dựng đặc trưng 126 số (giống 04_Trainv2) và dự đoán khung cuối
FPS_MODEL, WIN = 120, 128     # TCN chỉ nhìn 63 khung gần nhất, nên cửa sổ 128 khung cho kết quả y như nhìn cả clip
buf = collections.deque(maxlen=WIN)          # mỗi phần tử (21,3); NaN = mất tay
state = {"prev_lm": None, "prev_t": None}

def push(lm, t):
    """Thêm một khung webcam (lm hoặc None nếu mất tay, t tính bằng giây) vào buf, chèn các khung nội suy 120 fps."""
    prev, pt = state["prev_lm"], state["prev_t"]
    n = 1 if pt is None else max(1, round((t - pt) * FPS_MODEL))
    if lm is None:
        buf.extend([np.full((21, 3), np.nan, np.float32)] * n)
    elif prev is None:
        buf.append(lm)
    else:
        buf.extend(prev + (lm - prev) * (k / n) for k in range(1, n + 1))
    state["prev_lm"], state["prev_t"] = lm, t

def make_feat(lm):
    """lm: (T,21,3) -> (T,126) float32, đúng như hàm feat() ở 04_Trainv2 (cổ tay làm gốc, chia cỡ bàn tay, thêm vận tốc)."""
    lm = lm - lm[:, :1]
    lm = lm / np.linalg.norm(lm[:, 9], axis=1)[:, None, None]
    pos = lm.reshape(len(lm), 63)
    vel = np.vstack([np.zeros((1, 63)), np.diff(pos, axis=0)])
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32)

def predict():
    """Trả về (lớp, độ tin cậy) cho khung cuối, hoặc None nếu khung cuối không có tay."""
    lm = np.stack(buf)
    if np.isnan(lm[-1, 0, 0]):
        return None
    with torch.no_grad():
        p = net(torch.from_numpy(make_feat(lm))[None].to(dev)).softmax(-1)[0, -1]
    conf, c = p.max(0)
    return int(c), float(conf)
print("Sẵn sàng: nội suy", FPS_MODEL, "fps | cửa sổ", WIN, "khung")

Sẵn sàng: nội suy 120 fps | cửa sổ 128 khung


In [6]:
#cell 6: giao diện — màu theo lớp, chữ ĐEN trên nền màu (không dùng chữ trắng), khung tay + khung xương
COLORS = {0: (60, 60, 255), 1: (255, 150, 40), 2: (60, 210, 60)}   # BGR: paper = đỏ, rock = xanh dương, scissors = xanh lá
GRAY, YELLOW, BLACK = (190, 190, 190), (150, 255, 255), (0, 0, 0)
BONES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
         (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]

def put_label(img, text, x, y, bg, scale=0.8):
    """Chữ đen trên hộp màu; (x, y) là góc dưới-trái của hộp."""
    (w, h), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    cv2.rectangle(img, (x, y - h - base - 6), (x + w + 10, y + 2), bg, -1)
    cv2.putText(img, text, (x + 5, y - base), cv2.FONT_HERSHEY_SIMPLEX, scale, BLACK, 2, cv2.LINE_AA)

def draw_hand(img, img_lm, color, label, mirror=True):
    """Vẽ khung xương, khung bao quanh tay và nhãn (cùng màu) lên ảnh."""
    h, w = img.shape[:2]
    px, py = (img_lm[:, 0] * w).astype(int), (img_lm[:, 1] * h).astype(int)
    if mirror: px = w - 1 - px
    for a, b in BONES:
        cv2.line(img, (px[a], py[a]), (px[b], py[b]), color, 3, cv2.LINE_AA)
    for i in range(21):
        cv2.circle(img, (px[i], py[i]), 6, color, -1, cv2.LINE_AA)
        cv2.circle(img, (px[i], py[i]), 6, BLACK, 1, cv2.LINE_AA)
    x0, y0, x1, y1 = px.min() - 15, py.min() - 15, px.max() + 15, py.max() + 15
    cv2.rectangle(img, (x0, y0), (x1, y1), color, 3, cv2.LINE_AA)
    ly = y1 + 34 if (x0 < 220 and y0 < 125) else y0      # tránh đè lên khối FPS/ngưỡng ở góc trên trái
    put_label(img, label, max(x0, 0), min(max(ly, 32), h - 5), color)

def draw_hud(img, fps, thr):
    """FPS, ngưỡng và chú thích màu của 3 lớp."""
    put_label(img, f"FPS {fps:.0f}", 10, 35, YELLOW)
    put_label(img, f"Nguong >= {thr:.2f}", 10, 75, YELLOW)
    for i, name in enumerate(CLASSES):
        put_label(img, name, img.shape[1] - 125, img.shape[0] - 10 - 40 * (2 - i), COLORS[i])
print("Đã định nghĩa giao diện.")

Đã định nghĩa giao diện.


In [7]:
#cell 7: chạy webcam — q: thoát | [ ]: giảm/tăng ngưỡng tin cậy
CAM_ID, MIRROR, THR = 0, True, 0.70

cap = cv2.VideoCapture(CAM_ID, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))     # MJPG giúp webcam đạt 30 fps ở 640x480
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480); cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
assert cap.isOpened(), "Không mở được webcam (thử đổi CAM_ID sang 1)."

buf.clear(); state.update(prev_lm=None, prev_t=None)
fps, last = 0.0, time.perf_counter()
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        now = time.perf_counter()
        img_lm, model_lm = detect(frame, int((now - T0) * 1000))   # MediaPipe nhận ảnh gốc (không lật), giống lúc trích landmark
        push(model_lm, now)
        out = predict() if img_lm is not None else None

        disp = cv2.flip(frame, 1) if MIRROR else frame             # chỉ lật để hiển thị cho dễ chơi
        fps, last = 0.9 * fps + 0.1 / max(now - last, 1e-6), now
        if out is None:
            put_label(disp, "no hand", 10, 115, GRAY)
        else:
            c, conf = out
            sure = conf >= THR
            draw_hand(disp, img_lm, COLORS[c] if sure else GRAY,
                      f"{CLASSES[c]} {conf:.0%}" if sure else f"{CLASSES[c]}? {conf:.0%}", MIRROR)
        draw_hud(disp, fps, THR)

        cv2.imshow("RPS webcam - wide45_lownoise (q: thoat)", disp)
        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"): break
        if k == ord("["): THR = max(0.05, THR - 0.05)
        if k == ord("]"): THR = min(0.95, THR + 0.05)
finally:
    cap.release(); cv2.destroyAllWindows()
print("Đã đóng webcam.")

Đã đóng webcam.
